# Compare Benchmark Runs
This notebook demonstrates how you can analyze the differences between two benchmark runs of the same benchmark and find the tests that differ the most, which probably means that they require further analysis to figure out why they changed.

Several projects exist in the `examples` folder, but this notebook assumes we are working on the
JVM part of the `kotlin-multiplatform` project. But the same approach can be used for the other projects.

First, you need to run the benchmark twice. This can be done by running these commands from the root of the project:

```shell
> ./gradlew :examples:kotlin-multiplatform:jvmBenchmark
> ./gradlew :examples:kotlin-multiplatform:jvmBenchmark
```

Once it is completed, run this notebook, and it will automatically find the latest result.

In [1]:
%use serialization, dataframe, kandy

In [2]:
// Serialization classes matching the JMH-alike JSON format.
// We define these classes manually so we can keep `params` as a JsonObject, as it means we can handle them
// in a generic manner. If you benchmark have fixed params, using `"<jsonText>".deserializeThis()` is
// faster and easier.

@Serializable
public data class Benchmark(
    public val benchmark: String,
    public val mode: String,
    public val warmupIterations: Int,
    public val warmupTime: String,
    public val measurementIterations: Int,
    public val measurementTime: String,
    public val primaryMetric: PrimaryMetric,
    public val secondaryMetrics: Map<String, PrimaryMetric>,
    public val params: JsonObject? = null
)

@Serializable
public data class PrimaryMetric(
    public val score: Double,
    public val scoreError: Double,
    public val scoreConfidence: List<Double>,
    public val scorePercentiles: Map<String, Double>,
    public val scoreUnit: String,
    public val rawData: List<List<Double>>,
)

In [16]:
import java.nio.file.Files
import java.nio.file.attribute.BasicFileAttributes
import kotlin.io.path.*

val runsDir = notebook.workingDir.resolve("build/reports/benchmarks/main")
val resultsFile = runsDir.listDirectoryEntries()
    .filter { it.isDirectory() }
    .maxByOrNull { dir -> Files.readAttributes(dir, BasicFileAttributes::class.java).creationTime() }!!
    .resolve("js.json")

In [17]:
val json = Json { ignoreUnknownKeys = true }
val allResults = json.decodeFromString<List<Benchmark>>(resultsFile.readText())

// Split method name into (library, operation, scenario)
// e.g. "...JsonSerializationBenchmark.korlibs_encode_simple" -> korlibs / encode / simple
data class TidyRow(
    val library: String,
    val operation: String,
    val scenario: String,
    val label: String,        // "encode / simple"
    val score: Double,
    val errorLow: Double,
    val errorHigh: Double,
)

val scenarioOrder = listOf(
    "simple",
    "nested",
    "list",
    "nullable_full",
    "nullable_nulls",
    "fetchTimeViaNow",
    "thousandTimesMilliseconds",
    "arrayListGet",
    "arrayListGenericGet",
    "stringGet",
)

val tidy = allResults.mapNotNull { r ->
    val method = r.benchmark.substringAfterLast('.')  // e.g. "korlibs_encode_simple"
    val parts  = method.split("_", limit = 3)
    if (parts.size < 3) return@mapNotNull null
    val (lib, op, scenario) = parts
    if (lib !in listOf("korlibs", "kotlinx")) return@mapNotNull null
    TidyRow(
        library   = lib,
        operation = op,
        scenario  = scenario,
        label     = "$op / $scenario",
        score     = r.primaryMetric.score,
        errorLow  = r.primaryMetric.scoreConfidence[0],
        errorHigh = r.primaryMetric.scoreConfidence[1],
    )
}.sortedWith(compareBy({ it.operation }, { scenarioOrder.indexOf(it.scenario) }))

println("Parsed ${tidy.size} rows")
tidy.take(6).forEach { println(it) }

Parsed 32 rows
TidyRow(library=korlibs, operation=datastructure, scenario=arrayListGet, label=datastructure / arrayListGet, score=2865.1841494845357, errorLow=2819.0406074672997, errorHigh=2911.327691501772)
TidyRow(library=kotlinx, operation=datastructure, scenario=arrayListGet, label=datastructure / arrayListGet, score=3074.9963128491618, errorLow=3051.6412712645288, errorHigh=3098.3513544337948)
TidyRow(library=korlibs, operation=datastructure, scenario=arrayListGenericGet, label=datastructure / arrayListGenericGet, score=2894.494271099744, errorLow=2836.9613847596947, errorHigh=2952.027157439793)
TidyRow(library=kotlinx, operation=datastructure, scenario=arrayListGenericGet, label=datastructure / arrayListGenericGet, score=2980.877124010554, errorLow=2935.706188755309, errorHigh=3026.0480592657987)
TidyRow(library=korlibs, operation=datastructure, scenario=stringGet, label=datastructure / stringGet, score=273.7076594187959, errorLow=177.18273624178875, errorHigh=370.2325825958031)


In [18]:
// ── Plot 1: Grouped bar chart, encode + decode side by side ─────────────────
import org.jetbrains.kotlinx.kandy.util.color.Color

val palette = mapOf("korlibs" to Color.RED, "kotlinx" to Color.BLUE)

tidy.groupBy { it.operation }.forEach { (op, rows) ->
    rows.sortedBy { scenarioOrder.indexOf(it.scenario) }
        .toDataFrame()
        .plot {
            barsH {
                x("score") { axis.name = "Average time (µs)" }
                y("label") { axis.name = "" }
                fillColor("library") {
                    scale = categorical(
                        "korlibs" to Color.RED,
                        "kotlinx" to Color.BLUE
                    )
                }
                // Error bars via separate layer isn't in kandy barsH directly,
                // so add them as a point range on the same axes:
            }
            layout {
                title = "${op.replaceFirstChar { it.uppercaseChar() }}: korlibs vs kotlinx"
                size = 800 to ((50 * rows.size) + 120)
            }
        }.also { println(it) } // renders each plot inline
}

Plot(datasets=[NamedData(dataFrame=         score                               label library
 0 2865,184149        datastructure / arrayListGet korlibs
 1 3074,996313        datastructure / arrayListGet kotlinx
 2 2894,494271 datastructure / arrayListGenericGet korlibs
 3 2980,877124 datastructure / arrayListGenericGet kotlinx
 4  273,707659           datastructure / stringGet korlibs
 5  522,507943           datastructure / stringGet kotlinx
)], layers=[Layer(datasetIndex=0, geom=LetsPlotGeom(name=bar), mappings={Aes(name=x)=PositionalMapping(aes=Aes(name=x), columnID=score, parameters=LetsPlotPositionalMappingParametersContinuous(scale=org.jetbrains.kotlinx.kandy.ir.scale.PositionalDefaultScale@449609da, axis=Axis(name=Average time (µs), position=DEFAULT, min=null, max=null, breaks=null, labels=null, format=null, expand=null))), Aes(name=y)=PositionalMapping(aes=Aes(name=y), columnID=label, parameters=LetsPlotPositionalMappingParametersContinuous(scale=org.jetbrains.kotlinx.kandy.ir

In [19]:
// ── Plot 2: Ratio chart (korlibs ÷ kotlinx) ──────────────────────────────────
data class RatioRow(val label: String, val ratio: Double, val color: String)

val ratios = tidy
    .groupBy { "${it.operation}_${it.scenario}" }
    .mapNotNull { (_, rows) ->
        val k  = rows.firstOrNull { it.library == "korlibs" } ?: return@mapNotNull null
        val kx = rows.firstOrNull { it.library == "kotlinx" } ?: return@mapNotNull null
        RatioRow(
            label = "${k.operation} / ${k.scenario}",
            ratio = k.score / kx.score,
            color = if (k.score > kx.score) "korlibs slower" else "korlibs faster"
        )
    }
    .sortedByDescending { it.ratio }

ratios.toDataFrame().plot {
    barsH {
        x("ratio") { axis.name = "korlibs ÷ kotlinx  (>1 = korlibs slower)" }
        y("label") { axis.name = "" }
        fillColor("color") {
            scale = categorical(
                "korlibs slower" to Color.RED,
                "korlibs faster" to Color.GREEN
            )
        }
    }
    layout {
        title = "Performance ratio — korlibs vs kotlinx"
        size = 800 to ((50 * ratios.size) + 120)
    }
}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="SCAndN" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 800.0, 
 height: 870.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("SCAndN");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"Performance ratio — korlibs vs kotlinx"
},
"mapping":{
},
"data":{
"color":["korlibs slower","korlibs slower","korlibs slower","korlibs slower","korlibs slower","korlibs slower","korlibs slower","korlibs slower","korlibs slower","korlibs faster","korlibs faster","korlibs faster","korlibs faster","korlibs faster","korlibs faster"],
"label":["encode / nested","encode / list","encode / simple","encode / nullable_nulls","encode / nullable_full","decode / nested","decode / nullable_nulls","decode / simple","decode / list","datastructure / arrayListGenericGet","datastructure / arrayListGet","decode / nullable_full","time / thousandTimesMilliseconds","datastructure / stringGet","time / fetchTimeViaNow"],
"ratio":[2.0448075440947013,1.8959330874921079,1.645699168515098,1.6032440695062062,1.4323618121192891,1.2671704081062556,1.1371228151241684,1.129599421411956,1.0049009094546848,0.9710209950571232,0.9317683203430501,0.911389622329013,0.7967762145539193,0.5238344472160655,0.04682558385134758]
},
"ggsize":{
"width":800.0,
"height":870.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"name":"korlibs ÷ kotlinx (>1 = korlibs slower)",
"limits":[null,null]
},{
"aesthetic":"y",
"discrete":true,
"name":""
},{
"aesthetic":"fill",
"values":["#ee6666","#3ba272"],
"limits":["korlibs slower","korlibs faster"]
}],
"layers":[{
"mapping":{
"x":"ratio",
"y":"label",
"fill":"color"
},
"stat":"identity",
"orientation":"y",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data":{
}
}],
"data_meta":{
"series_annotations":[{
"type":"float",
"column":"ratio"
},{
"type":"str",
"column":"label"
},{
"type":"str",
"column":"color"
}]
},
"spec_id":"14"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 
 
 
 
 0.5 
 
 
 
 
 
 
 
 
 1 
 
 
 
 
 
 
 
 
 1.5 
 
 
 
 
 
 
 
 
 2 
 
 
 
 
 
 
 
 
 
 
 encode / nested 
 
 
 
 
 
 
 encode / list 
 
 
 
 
 
 
 encode / simple 
 
 
 
 
 
 
 encode / nullable_nulls 
 
 
 
 
 
 
 encode / nullable_full 
 
 
 
 
 
 
 decode / nested 
 
 
 
 
 
 
 decode / nullable_nulls 
 
 
 
 
 
 
 decod